###### Documentação parquet
https://spark.apache.org/docs/latest/sql-data-sources-parquet.html 

https://spark.apache.org/docs/latest/sql-data-sources-parquet.html

### Particionamento em Parquet ou Delta: Vantagens, Exemplos e Boas Práticas

O particionamento de dados em formatos como Parquet ou Delta consiste em dividir grandes conjuntos de dados em subdiretórios baseados em valores de uma ou mais colunas (ex: data, estado, categoria). Isso permite que consultas leiam apenas os arquivos relevantes, otimizando performance e reduzindo custos de leitura.

**Vantagens:**
- **Leitura eficiente:** Apenas partições relevantes são lidas, acelerando consultas.
- **Escalabilidade:** Facilita o processamento distribuído em grandes volumes de dados.
- **Gerenciamento:** Organiza dados de forma lógica, facilitando manutenção e arquivamento.

**Exemplo de uso em Spark:**
python
df.write.partitionBy("coluna_particao").parquet("/caminho/para/parquet_particionado")

Ou para Delta:
python
df.write.partitionBy("coluna_particao").format("delta").save("/caminho/para/delta_particionado")


**Volume recomendado:**  
Particionamento é recomendado para tabelas a partir de algumas centenas de milhares de linhas ou arquivos acima de 1GB. Para volumes pequenos, particionar pode não trazer ganhos e até prejudicar a performance.

**Cuidado com excesso de partições:**  
Particionar demais pode gerar muitos arquivos pequenos ("small files problem"), aumentando o tempo de leitura e o overhead do metastore. Isso pode fazer com que consultas SQL ou Spark SQL realizem um scan completo do dataset, anulando os ganhos de performance.

**Boas práticas:**
- Escolha colunas com baixa cardinalidade para particionar (ex: ano, mês, estado).
- Evite particionar por colunas com muitos valores únicos (ex: IDs).
- Monitore o número de arquivos e o tamanho médio das partições.

**Referências:**
- [Documentação Parquet](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html)
- [Documentação Delta Lake](https://docs.delta.io/latest/delta-batch.html#partitioning)

In [0]:
'''
path
└── to
    └── table
        ├── gender=male
        │   ├── ...
        │   │
        │   ├── country=US
        │   │   └── data.parquet
        │   ├── country=CN
        │   │   └── data.parquet
        │   └── ...
        └── gender=female
            ├── ...
            │
            ├── country=US
            │   └── data.parquet
            ├── country=CN
            │   └── data.parquet
            └── ...
'''

In [0]:
df_parquet=spark.read.parquet("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet/")
df_parquet.display()

In [0]:
#verificando dados distintos de colunas com spark sql Classificacao_da_Ocorrência

display(df_parquet.select("Classificacao_da_Ocorrência").distinct())

In [0]:
#verificando dados distintos de colunas com spark sql UF
display(df_parquet.select("UF").distinct())



Salvar em particionamento UF, outra para Ocorrencias e outro usando os 2

Salvar partition = UF

In [0]:
(
df_parquet.write
  .format("parquet")
  .partitionBy("UF")
  .mode("overwrite")
  .save("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_UF/")
)

In [0]:
#ver arquivo particionado
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_UF/"))

In [0]:
#Descendo mais um nivel 
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_UF/UF=MG/"))


Partition = Ocorrencia

In [0]:
(
df_parquet.write
  .format("parquet")
  .partitionBy("Classificacao_da_Ocorrência")
  .mode("overwrite")
  .save("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_ocorrencia/")
)

In [0]:
#ver arquivo particionado
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_ocorrencia/"))

In [0]:
#ver arquivo particionado subnivel
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_ocorrencia/Classificacao_da_Ocorrência=Incidente Grave/"))

In [0]:
#lendo pasta geral
df_uf_geral=spark.read.parquet("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_UF/")
display(df_uf_geral)


In [0]:
from pyspark.sql.functions import lit
# ler apenas uma partição especifica
df_uf_mg = spark.read.parquet("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_partition_UF/UF=MG/")

df_uf_mg=df_uf_mg.withColumn("UF", lit("MG"))

display(df_uf_mg)

######Salvando em mais de 1 partição 

In [0]:
#simulando o responsavel de Mg onde a Classificacao da Ocorrência seja igual a Acidente trabalharia com o arquivo 
#dbfs:/FileStore/tables/Anac/parquet_Multiparticionado/UF=MG/Classificacao_da_Ocorrência=Acidente/
#e nao todo o arquivo original , ganhando performace na leitura 

In [0]:
(
df_parquet.write
  .format("parquet")
  .partitionBy("UF","Classificacao_da_Ocorrência")
  .mode("overwrite")
  .save("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_multipartition/")
)

In [0]:
#ver arquivo particionado subnivel
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_multipartition/"))

In [0]:
#ver arquivo particionado subnivel2
display(dbutils.fs.ls("/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_multipartition/UF=MG/"))

Dados Acidente do analista de MG

In [0]:
df_minas_grave=spark.read.parquet('/Volumes/curso_databricks/aula/aula_volume/Anac/parquet_multipartition/UF=MG/Classificacao_da_Ocorrência=Incidente Grave/')
df_minas_grave.display()

In [0]:
%sql
-- exemplo de consulta otimizada
SELECT * FROM suatabela 
where UF = 'MG'
and Classificacao_da_Ocorrência='Incidente Grave'